# LEGEND poster — reproducible analysis workbench

**Provisional physics benchmark.** Run `poster_setup.ipynb` to change and save
assumptions, then run this notebook. Physics comes from existing WIMP-LEGEND
APIs. Figures consume saved numerical outputs, not pilot sensitivity tables.

The original notebook is preserved byte-for-byte at
[historical/poster_workbench_pilot.ipynb](historical/poster_workbench_pilot.ipynb).
Its assets are historical references, not validated predictions.


In [ ]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd().resolve()
while not (ROOT / 'pyproject.toml').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('Open this notebook inside the WIMP-LEGEND checkout')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
from IPython.display import display, Image
from analyses.legend_poster import load_config, save_config, validate_config, resolve_profile
CONFIG_PATH = Path(os.environ.get('WIMP_POSTER_CONFIG', ROOT / 'analyses/legend_poster/config.json'))
print('Configuration:', CONFIG_PATH)


## Load the saved configuration

In [ ]:
configuration = load_config(CONFIG_PATH)
profile = os.environ.get('WIMP_POSTER_PROFILE', configuration['execution']['default_profile'])
resolved = resolve_profile(configuration, profile)
print(resolved['status'])
print('Profile:', resolved['execution']['resolved_label'])
print('SI rate engine:', resolved['signal'].get('si_rate_engine', 'generic'))
display(resolved['analysis'])
display(resolved['provenance']['notes'])


## Target mixtures and coefficient conventions

Rates are computed for each isotope with the accepted AFH datasets and combined
with the configured mixture's **bulk mass weights**. Bare masses enter
kinematics; neutral atomic masses enter per-kg detector counts. Solar CEvNS is
also summed isotope by isotope; solar ES uses the exact total Ge atom count.

O1 uses the physical proton SI adapter. O8/O11 are single-operator limits on
`|c_i^p|` in GeV^-2 with the saved spin and neutron/proton ratio. They are **not
SI cross sections**. The frozen half-isospin convention is unchanged.


In [ ]:
from analyses.legend_poster.models import isotope_weights
for name in resolved['target']['presets']:
    print(name)
    display(isotope_weights(resolved, name))
display({k: resolved['signal'][k] for k in ('convention_id','j_chi','fn_over_fp',
    'coefficient_unit','sigma_p_ref_cm2','coefficient_ref_GeV_minus2','m_N_reference_GeV')})


## Run or reuse numerical outputs

The first run evaluates core recoil rates, integrates every observed-energy bin,
and checks refinement. The disk cache is keyed by physics inputs, source hashes,
software versions and numerical settings. Exposure and plot-only edits reuse
expensive source calculations. No pickle cache is used. A failed convergence
check raises instead of producing an accepted result.


In [ ]:
from analyses.legend_poster import run_analysis
run = run_analysis(configuration, profile=profile,
    cache_dir=os.environ.get('WIMP_POSTER_CACHE'),
    output_dir=os.environ.get('WIMP_POSTER_RUNS'))
print('Saved run:', run['path'])
print('Run ID:', run['manifest']['run_id'])
print('Code commit:', run['manifest']['source']['git_commit'])
print('Cache hits/misses:', run['manifest']['cache']['hits'], run['manifest']['cache']['misses'])


## Component spectra and threshold features

Rates below are integrated counts divided by each bin's own width, in
`events/(kg year keV observed)`. Lines use detector bin probabilities; truth
continua are reintegrated. Already-observed inputs are never refolded.

The next cell reloads only saved arrays and current plotting settings, so it
can be rerun after styling edits without recomputing physics.


In [ ]:
from analyses.legend_poster.plotting import plot_run
plot_style = load_config(CONFIG_PATH)['plotting']
figures = plot_run(run, plotting=plot_style)
for scenario in run['manifest']['resolved_configuration']['analysis']['scenarios']:
    display(Image(filename=str(run['path'] / 'figures' / ('background_' + scenario + '.png'))))


## SI and NREFT sensitivities

Counting limits use a threshold-anchored ROI containing at least 95% of the
signal **inside the configured observed window**, then median exact known-B
Poisson CLs at the saved confidence level. ROI edges are saved per mass/operator.
SI scales as `sigma_upper = mu_upper * sigma_ref`; single-operator coefficients
scale as `|c_upper| = sqrt(mu_upper) * |c_ref|`.

Optional `binned_asimov` uses the separate asymptotic profile backend over the
full above-threshold window; it is neither exact CLs nor a sparse-count coverage
claim. External curves retain their own confidence/construction assumptions and
are frozen snapshots, not guaranteed latest 2026 exclusions. Projections are dashed.


In [ ]:
for operator in run['manifest']['resolved_configuration']['signal']['operators']:
    display(Image(filename=str(run['path'] / 'figures' / f'limit_O{operator}.png')))
if 1 in run['manifest']['resolved_configuration']['signal']['operators']:
    display(Image(filename=str(run['path'] / 'figures' / 'exposure_comparison.png')))
display(run['limits'][:12])


## Inspect counts, convergence and historical comparison

Historical/revised curves share the same signal, exposure, ROI and statistical
settings. The historical background is an **already-observed frozen template**:
its original response/selection is retained, not recalibrated to arbitrary new
detector assumptions. This is a count-template comparison, not a controlled
single-physics-correction measurement. The original pilot limit CSV is not used.


In [ ]:
checks = run['convergence']
assert all(item['passed'] for item in checks)
print('Maximum total refinement change:', max(item['total_relative'] for item in checks))
print('Maximum significant-bin refinement change:', max(item['significant_bin_relative'] for item in checks))
display(checks[:12])
paired = resolved['background']['historical']['pair_with']
display([row for row in run['limits'] if row['scenario'] == paired][:12])
display(run['manifest']['numerical_notes'])


## Reproducibility and remaining scientific checks

Each run contains resolved JSON, Git/source/data hashes, integrated isotope and
component counts, limit construction/units, refinement diagnostics and output
checksums. Each figure set has its own provenance/hash manifest. See
`analyses/legend_poster/README.md` and `TESTER_HANDOFF.md`.

Before publication, replace hypothetical enrichment, exposure history, LAr
transfer/veto and detector calibration; audit missing solar components
(pp/B8 only), nuclear-response validity, source mass uncertainties and CL/model
assumptions. Neither pilot nor new benchmark is a validated LEGEND prediction.
The inherited core workspace-boundary and private-cache serialization findings
remain open; this analysis neither repairs nor silently hides them.
